%md
#SILVER_TRANSFORMATION for bronze.orders

In [0]:
from pyspark.sql.functions import row_number, col, current_timestamp
from pyspark.sql.window import Window

In [0]:
df = spark.table("bronze.order_items").select("*")

In [0]:
df.limit(5).display()

In [0]:
df.count()

### STEP 1: remove duplicates

In [0]:
w = Window.partitionBy("order_id", "order_item_id").orderBy(
    col("shipping_limit_date").desc(),
    col("_load_timestamp").desc()      
)

In [0]:
df = df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")
df.count()

### Step 2: casting

In [0]:
df = df.withColumn("price", col("price").cast("decimal(10,2)")) \
       .withColumn("freight_value", col("freight_value").cast("decimal(10,2)"))

### Step 3: null checks

In [0]:
df = df.filter(col("order_id").isNotNull() & col("order_item_id").isNotNull())

### Step 4: write to SILVER

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
(
    df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable("silver.order_items")
)

In [0]:
cnt = spark.table("silver.order_items").count()
print(f"silver.order_items: {cnt:,} rows")